# Regression - 배터리 수명 예측

이 노트북은 **초기 100 사이클 데이터만으로 배터리의 총 수명(`cycle_life`)을 예측**하는 회귀 실험입니다.

## 실험 설정
- **Train**: Batch 1
- **Test**: Batch 2
- **Target**: `cycle_life` — EOL(80% SOH 도달)까지의 총 사이클 수
- **Metric**: `MAPE (%)`
- **Model**: `LGBMRegressor` (LightGBM)
- **원논문 목표 성능**: 9.1% (MAPE)

## 성능 보고 형식

| Index | 설명 |
|---|---|
| Train (Batch 1 CV) | Batch 1 내 train subset에서 Cross-Validation 평균 성능 |
| Valid (Batch 1 Hold-out) | Batch 1 내 Hold-out 검증 성능 |
| Test (Batch 2) | Batch 2 최종 평가 성능 |
| Gap (Train-Valid) | Train 과적합 여부 확인 |
| Gap (Valid-Test) | 배치 간 일반화 차이 확인 |
| Gap (Target-Test) | 원논문 성능(9.1%) 대비 차이 |

## Valid를 CV가 아닌 Hold-out으로 사용하는 이유
- 배터리 데이터는 **셀 단위로 독립적**이며, 각 셀이 서로 다른 충전 프로토콜(C-rate)로 실험됨
- CV를 적용하면 동일 프로토콜 셀이 train/valid에 나뉘어 **데이터 누수(leakage)** 위험 잔존
- Hold-out은 셀 단위 분리를 명확히 보장하며, **배치 간 일반화**를 평가하는 이번 프로젝트 구조에 더 적합

## 참고
- 현재 데이터셋은 **전처리와 결측치 처리가 이미 완료된 상태**라고 가정합니다.
- 따라서 이 노트북에서는 **추가 전처리 없이** 바로 학습/평가만 수행합니다.
- 이번 버전은 **프로토콜 그룹 분리 없이**, 현재 주어진 feature dataset 구조에 맞춰 **일반 Hold-out 방식**으로 평가합니다.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.metrics import make_scorer
from lightgbm import LGBMRegressor

In [2]:
# =========================
# 1. Config
# =========================
FEATURES = [
    "delta_log_var",
    "QD_cv",
    "effective_C",
    "temp_ir_interaction",
]
TARGET = "cycle_life"
TARGET_SCORE = 9.1   # 원논문 Regression MAPE (%)
RANDOM_STATE = 42

# 파일 경로 후보
BATCH1_CANDIDATES = [
    Path("batch1_model_df.csv"),
    Path("data/processed/batch1_model_df.csv"),
    Path("/Users/skax/skala/ess-battery-project/data/processed/batch1_model_df.csv"),
]

BATCH2_CANDIDATES = [
    Path("batch2_model_df.csv"),
    Path("data/processed/batch2_model_df.csv"),
    Path("/Users/skax/skala/ess-battery-project/data/processed/batch2_model_df.csv"),
]

def resolve_existing_path(candidates):
    for p in candidates:
        if p.exists():
            return str(p)
    raise FileNotFoundError(
        "CSV 파일을 찾지 못했습니다. 아래 후보 경로를 확인하세요:\n"
        f"- batch1: {[str(p) for p in BATCH1_CANDIDATES]}\n"
        f"- batch2: {[str(p) for p in BATCH2_CANDIDATES]}"
    )

BATCH1_PATH = resolve_existing_path(BATCH1_CANDIDATES)
BATCH2_PATH = resolve_existing_path(BATCH2_CANDIDATES)

print("BATCH1_PATH =", BATCH1_PATH)
print("BATCH2_PATH =", BATCH2_PATH)

BATCH1_PATH = /Users/skax/skala/ess-battery-project/data/processed/batch1_model_df.csv
BATCH2_PATH = /Users/skax/skala/ess-battery-project/data/processed/batch2_model_df.csv


In [3]:
# =========================
# 2. Metric
# =========================
def mape(y_true, y_pred):
    """Mean Absolute Percentage Error (%)"""
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

mape_scorer = make_scorer(mape, greater_is_better=False)

In [4]:
# =========================
# 3. 데이터 로드
# =========================
batch1 = pd.read_csv(BATCH1_PATH)
batch2 = pd.read_csv(BATCH2_PATH)

print("batch1 shape:", batch1.shape)
print("batch2 shape:", batch2.shape)

batch1 shape: (46, 33)
batch2 shape: (39, 33)


In [5]:
# =========================
# 4. 컬럼 확인
# =========================
print("batch1 columns:", batch1.columns.tolist())
print("batch2 columns:", batch2.columns.tolist())

required_cols = FEATURES + [TARGET]
missing_b1 = [c for c in required_cols if c not in batch1.columns]
missing_b2 = [c for c in required_cols if c not in batch2.columns]

if missing_b1:
    raise ValueError(f"batch1에 없는 필수 컬럼: {missing_b1}")
if missing_b2:
    raise ValueError(f"batch2에 없는 필수 컬럼: {missing_b2}")

print("필수 컬럼 확인 완료")

batch1 columns: ['cell_id', 'QD_mean', 'QD_std', 'IR_mean', 'IR_std', 'Tavg_mean', 'Tavg_std', 'C_rate_1', 'ratio', 'C_rate_2', 'effective_C', 'slope_early', 'slope_mid', 'slope_late', 'slope_diff', 'QD_cv', 'IR_cv', 'temp_ir_interaction', 'delta_mean', 'delta_std', 'delta_var', 'delta_log_var', 'delta_min', 'delta_max', 'delta_range', 'delta_area', 'delta_skew', 'delta_kurtosis', 'delta_v_2_0', 'delta_v_2_8', 'delta_v_3_0', 'delta_28_30_mean', 'cycle_life']
batch2 columns: ['cell_id', 'QD_mean', 'QD_std', 'IR_mean', 'IR_std', 'Tavg_mean', 'Tavg_std', 'C_rate_1', 'ratio', 'C_rate_2', 'effective_C', 'slope_early', 'slope_mid', 'slope_late', 'slope_diff', 'QD_cv', 'IR_cv', 'temp_ir_interaction', 'delta_mean', 'delta_std', 'delta_var', 'delta_log_var', 'delta_min', 'delta_max', 'delta_range', 'delta_area', 'delta_skew', 'delta_kurtosis', 'delta_v_2_0', 'delta_v_2_8', 'delta_v_3_0', 'delta_28_30_mean', 'cycle_life']
필수 컬럼 확인 완료


In [6]:
# =========================
# 5. X, y 분리
# =========================
X = batch1[FEATURES]
y = batch1[TARGET]

X_test = batch2[FEATURES]
y_test = batch2[TARGET]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

X shape: (46, 4)
y shape: (46,)
X_test shape: (39, 4)
y_test shape: (39,)


In [7]:
# =========================
# 6. Batch 1 Hold-out 분리
# =========================
# 배터리 셀 단위 독립성 보장 및 데이터 누수 방지를 위해 Hold-out 방식 사용
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
)

print("X_train:", X_train.shape)
print("X_valid:", X_valid.shape)

X_train: (36, 4)
X_valid: (10, 4)


In [8]:
# =========================
# 7. Train (Batch 1 CV)
#    - Hold-out으로 나눈 train subset 내부에서 CV 수행
# =========================
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

model_cv = LGBMRegressor(
    learning_rate=0.05,
    max_depth=3,
    n_estimators=300,
    min_child_samples=5,
    random_state=RANDOM_STATE,
    verbose=-1,           # LightGBM 학습 로그 억제
)

cv_scores = cross_val_score(
    model_cv,
    X_train,
    y_train,
    cv=cv,
    scoring=mape_scorer,
)

train_cv_mape = -cv_scores.mean()
print("CV fold MAPE (%):", np.round(-cv_scores, 4))
print("Train (Batch 1 CV) MAPE:", round(train_cv_mape, 4))

CV fold MAPE (%): [13.5234 16.7635  7.316   9.8123 11.5871]
Train (Batch 1 CV) MAPE: 11.8005


In [9]:
# =========================
# 8. Valid / Test 평가
# =========================
final_model = LGBMRegressor(
    learning_rate=0.05,
    max_depth=3,
    n_estimators=300,
    min_child_samples=5,
    random_state=RANDOM_STATE,
    verbose=-1,
)

final_model.fit(X_train, y_train)

valid_pred = final_model.predict(X_valid)
test_pred  = final_model.predict(X_test)

valid_mape = mape(y_valid, valid_pred)
test_mape  = mape(y_test,  test_pred)

print("Valid MAPE:", round(valid_mape, 4))
print("Test  MAPE:", round(test_mape,  4))

Valid MAPE: 8.7025
Test  MAPE: 26.4071


In [10]:
# =========================
# 9. Gap 계산 및 결과 테이블
# =========================
gap_train_valid = valid_mape - train_cv_mape  # 음수면 valid가 더 좋음 → 과적합 낮음
gap_valid_test  = test_mape  - valid_mape     # 양수면 test 성능 저하 → 배치 간 차이
gap_target_test = test_mape  - TARGET_SCORE   # 양수면 논문보다 나쁨, 음수면 더 좋음

result_df = pd.DataFrame({
    "Index": [
        "Train (Batch 1 CV)",
        "Valid (Batch 1 Hold-out)",
        "Test (Batch 2)",
        "Gap (Train-Valid)",
        "Gap (Valid-Test)",
        "Gap (Target-Test)",
    ],
    "MAPE (%)": [
        train_cv_mape,
        valid_mape,
        test_mape,
        gap_train_valid,
        gap_valid_test,
        gap_target_test,
    ],
})

result_df.round(4)

,Index,MAPE (%)
0,Train (Batch 1 CV),11.8005
1,Valid (Batch 1 Hold-out),8.7025
2,Test (Batch 2),26.4071
3,Gap (Train-Valid),-3.0980
4,Gap (Valid-Test),17.7046
5,Gap (Target-Test),17.3071


In [11]:
# =========================
# 10. 결과 해석용 출력
# =========================
print(result_df.round(4).to_string(index=False))

print("\n[해석 가이드]")
print("- Train (Batch 1 CV)      : 학습 데이터 내부 CV 평균 성능 (낮을수록 좋음)")
print("- Valid (Batch 1 Hold-out): Batch 1 내 독립 검증 성능 (낮을수록 좋음)")
print("- Test (Batch 2)          : Batch 2 최종 일반화 성능 (낮을수록 좋음)")
print("- Gap (Train-Valid)       : 음수 or 낮을수록 과적합 낮음")
print("- Gap (Valid-Test)        : 양수 클수록 배치 간 일반화 저하 가능성 높음")
print(f"- Gap (Target-Test)       : 양수면 논문 목표({TARGET_SCORE}%)보다 성능이 나쁨, 음수면 달성")

                   Index  MAPE (%)
      Train (Batch 1 CV)   11.8005
Valid (Batch 1 Hold-out)    8.7025
          Test (Batch 2)   26.4071
       Gap (Train-Valid)   -3.0980
        Gap (Valid-Test)   17.7046
       Gap (Target-Test)   17.3071

[해석 가이드]
- Train (Batch 1 CV)      : 학습 데이터 내부 CV 평균 성능 (낮을수록 좋음)
- Valid (Batch 1 Hold-out): Batch 1 내 독립 검증 성능 (낮을수록 좋음)
- Test (Batch 2)          : Batch 2 최종 일반화 성능 (낮을수록 좋음)
- Gap (Train-Valid)       : 음수 or 낮을수록 과적합 낮음
- Gap (Valid-Test)        : 양수 클수록 배치 간 일반화 저하 가능성 높음
- Gap (Target-Test)       : 양수면 논문 목표(9.1%)보다 성능이 나쁨, 음수면 달성
